In [3]:
from config import LLM_MODEL
print(LLM_MODEL)

qwen2.5:7b


In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd
from config import DOSSIER_CP, QUESTIONS, TOP_K
from src.extraction_cp.extraction import analyser_cp
from src.retrieval.retrieval import ouvrir_store, rechercher

store = ouvrir_store()
questions = json.loads(QUESTIONS.read_text(encoding="utf-8"))

# Le contrat de chaque CP, calculé une seule fois (6 extractions, pas 25)
contrat_de = {}
for nom_cp in sorted({q["cp"] for q in questions}):
    _, _, contrat_de[nom_cp] = analyser_cp(DOSSIER_CP / nom_cp)
print(contrat_de)

{'cp_01_luko_etendue.pdf': 'luko_hce_2504', 'cp_02_luko_minimum.pdf': 'luko_hmin_2504', 'cp_03_sogessur_F190315.pdf': 'sg_F190315_2025-04', 'cp_04_gmf_domopass.pdf': 'gmf_domopass_2023-01', 'cp_05_matmut_residence_principale.pdf': 'matmut_rp', 'cp_06_axa_hors_base.pdf': None}


In [2]:
lignes = []
for q in questions:
    # Les pages attendues dans les CG (on ignore la CP, qui n'est pas cherchée par le RAG)
    attendues = {(s["fichier"], p) for s in q["sources"]
                 if not s["fichier"].startswith("cp_") for p in s["pages"] if isinstance(p, int)}
    if not attendues:          # questions CP, HORS, ou pages non remplies : pas de Recall à calculer
        continue

    passages = rechercher(store, q["question"], contrat_de[q["cp"]], k=TOP_K)
    trouvees = {(p.metadata["fichier"], p.metadata["page"]) for p in passages}

    communes = attendues & trouvees
    lignes.append({"id": q["id"], "type": q["type"],
                   "hit@5": len(communes) > 0,
                   "recall@5": len(communes) / len(attendues),
                   "attendues": sorted(p for _, p in attendues),
                   "trouvees": [p.metadata["page"] for p in passages]})

resultats = pd.DataFrame(lignes)
print(f"Hit@5    : {resultats['hit@5'].mean():.0%}")
print(f"Recall@5 : {resultats['recall@5'].mean():.0%}")
resultats

Hit@5    : 81%
Recall@5 : 51%


,id,type,hit@5,recall@5,attendues,trouvees
0,Q01,CG,True,1.000000,[67],"[68, 67, 92, 25, 23]"
1,Q02,CG,True,0.333333,"[18, 19, 20, 58, 67, 68]","[18, 34, 1, 19, 38]"
2,Q03,CG,True,1.000000,"[30, 31]","[31, 31, 30, 38, 30]"
3,Q05,CG,True,1.000000,[16],"[16, 15, 16, 16, 15]"
4,Q06,CG,True,1.000000,"[37, 38]","[37, 37, 35, 37, 38]"
5,Q07,CG,True,0.500000,"[87, 88]","[88, 88, 88, 89, 96]"
6,Q08,CG,True,1.000000,"[11, 12]","[12, 11, 83, 84, 84]"
7,Q09,CG,False,0.000000,"[27, 28, 29]","[52, 23, 30, 2, 16]"
8,Q10,CG,True,0.500000,"[50, 51]","[51, 1, 18, 57, 16]"
9,Q16,CG+CP,True,0.428571,"[12, 23, 24, 25, 26, 67, 94]","[47, 12, 23, 36, 24]"


In [3]:
resultats.to_csv("../../results/retrieval_v1_baseline.csv", index=False)